# Prerequisites

In [1]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

--2026-09-17 10:12:48--  https://www.gutenberg.org/ebooks/103.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/103/pg103.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-17 10:12:48--  https://www.gutenberg.org/cache/epub/103/pg103.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 403712 (394K) [text/plain]
Saving to: ‘around_the_world_in_80_days.txt’

around_the_world_in 100%[===================>] 394.25K  --.-KB/s    in 0.09s   

2026-09-17 10:12:48 (4.06 MB/s) - ‘around_the_world_in_80_days.txt’ saved [403712/403712]



# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [2]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [3]:
# Defind the rdd
rdd = sc.textFile('/content/around_the_world_in_80_days.txt')

In [4]:
# view the first x lines of the rdd
rdd.take(20)

['The Project Gutenberg eBook of Around the World in Eighty Days',
 '    ',
 'This eBook is for the use of anyone anywhere in the United States and',
 'most other parts of the world at no cost and with almost no restrictions',
 'whatsoever. You may copy it, give it away or re-use it under the terms',
 'of the Project Gutenberg License included with this eBook or online',
 'at www.gutenberg.org. If you are not located in the United States,',
 'you will have to check the laws of the country where you are located',
 'before using this eBook.',
 '',
 'Title: Around the World in Eighty Days',
 '',
 'Author: Jules Verne',
 '',
 'Translator: George M. Towle',
 '',
 '',
 '        ',
 'Release date: January 1, 1994 [eBook #103]',
 '                Most recently updated: October 29, 2024']

In [5]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [6]:
# Note and explain the output of the below command
words

PythonRDD[3] at RDD at PythonRDD.scala:57

double-click and add explanation here :

Calling words returns a driver-side reference object PythonRDD, not the actual text data.

This behavior illustrates Spark's lazy evaluation: flatMap is a transformation, meaning Spark only records the operation in its Directed Acyclic Graph DAG without triggering immediate execution. Computations across partitions are only scheduled and materialized when an action such as take or collect is called.

<ADD EXPLANATION HERE>

In [7]:
# Note and explain the output of the following command, focusing on the difference with the
# above command

words.collect()[:10]

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty']

The previous command words only printed an RDD object because flatMap is a transformation. Spark uses lazy evaluation, so it does not process the data until an action is called.  On the other hand, words.collect() is an action. It forces Spark to run the job and brings all the words from the workers back to the driver as a Python list. We should be careful with collect() on very large datasets because it can crash the memory.

In [24]:
# nicer print
for w in words.collect()[:10]:
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)


In [9]:
# Print first x words
words.take(20)

['The',
 'Project',
 'Gutenberg',
 'eBook',
 'of',
 'Around',
 'the',
 'World',
 'in',
 'Eighty',
 'Days',
 '',
 '',
 '',
 '',
 '',
 'This',
 'eBook',
 'is',
 'for']

In [10]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
rdd.flatMap?

The flatMap function takes each line of text and splits it into multiple words using the space character. It then flattens the result so that we get a single RDD of words instead of a list of lists. If we used map we would have a list of words for each line.

In [11]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect()[:10]:
    print(w)

('The', 1)
('Project', 1)
('Gutenberg', 1)
('eBook', 1)
('of', 1)
('Around', 1)
('the', 1)
('World', 1)
('in', 1)
('Eighty', 1)


In [12]:
# a. count the occurence of each word

# We group by key (the word) and add the counts together
word_counts = words.reduceByKey(lambda a, b: a + b)

# Display the first 10 results to check the counts
word_counts.take(10)

[('Gutenberg', 60),
 ('eBook', 6),
 ('of', 1875),
 ('Around', 4),
 ('', 2193),
 ('for', 407),
 ('use', 16),
 ('anyone', 6),
 ('United', 23),
 ('States', 10)]

In [13]:
# b. a common first step in text analysis, change all capital letters to lower case

# Convert every word to lowercase using map
lower_words = rdd.flatMap(lambda lines: lines.split(" ")).map(lambda word: word.lower())

# Check the first 10 lowercase words
lower_words.take(10)

['the',
 'project',
 'gutenberg',
 'ebook',
 'of',
 'around',
 'the',
 'world',
 'in',
 'eighty']

In [14]:
# c. eliminate the stop words.

# Define a set of common words to remove
stopwords = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "as", "is", "was", "are", "were", "it",
    "this", "that", "he", "she", "they", "i", "you", "we", "his", "her",
    "their", "my", "your", "our", "be", "been", "had", "have", "has", "not"
}

# Keep only the words that are not in the stopwords set
no_stopwords = lower_words.filter(lambda word: word not in stopwords)

# Display the first 10 filtered words
no_stopwords.take(10)

['project',
 'gutenberg',
 'ebook',
 'around',
 'world',
 'eighty',
 'days',
 '',
 '',
 '']

In [15]:
# d. sort in alphabetical order

# Map each word to (word, 1), count with reduceByKey, and sort alphabetically
alpha_sorted = (
    no_stopwords
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortByKey()
)

# Display the first 10 words in alphabetical order
alpha_sorted.take(10)

[('', 2193),
 ('#103]', 1),
 ('#516,', 1),
 ('$5,000)', 1),
 ('&c.,', 1),
 ('($1', 1),
 ('(862)', 1),
 ('(a)', 1),
 ('(and', 1),
 ('(any', 1)]

In [16]:
# e. sort descending by word frequency

# Sort by count (index 1 of the tuple) in descending order
freq_sorted = (
    no_stopwords
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

# Display the top 10 most frequent words
freq_sorted.take(10)

[('', 2193),
 ('which', 490),
 ('mr.', 373),
 ('fogg', 365),
 ('would', 274),
 ('phileas', 250),
 ('passepartout', 239),
 ('him', 183),
 ('who', 182),
 ('if', 170)]

In [17]:
# f. remove punctuations and blank spaces
import string

# Define full pipeline function chaining all previous steps
def clean_text_pipeline(raw_rdd):
    # Set of stopwords to filter out
    stopwords = {
        "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
        "of", "with", "by", "from", "as", "is", "was", "are", "were", "it",
        "this", "that", "he", "she", "they", "i", "you", "we", "his", "her",
        "their", "my", "your", "our", "be", "been", "had", "have", "has", "not"
    }

    return (
        raw_rdd
        # Split lines into words
        .flatMap(lambda line: line.split(" "))
        # Lowercase and strip punctuation characters
        .map(lambda word: word.lower().strip(string.punctuation))
        # Filter out empty strings and stopwords
        .filter(lambda word: len(word) > 0 and word not in stopwords)
        # Create key-value pairs (word, 1)
        .map(lambda word: (word, 1))
        # Aggregate word counts
        .reduceByKey(lambda a, b: a + b)
    )

# Run pipeline on the book RDD and sort descending by frequency
final_counts = clean_text_pipeline(rdd).sortBy(lambda x: x[1], ascending=False)

# Display top 10 words
final_counts.take(10)

[('fogg', 577),
 ('which', 515),
 ('passepartout', 392),
 ('mr', 373),
 ('him', 314),
 ('would', 278),
 ('phileas', 250),
 ('fix', 228),
 ('who', 199),
 ('said', 194)]

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [18]:
# Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # Map each record to (name, (age, 1)) to track both age and count
  .map(lambda x: (x[0], (x[1], 1)))
  # Add total ages and counts together for each unique name
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # Divide total age by count to compute average age per name
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

# Print results
print(agesRDD.collect())

[('Brooke', 22.5), ('Denny', 31.0), ('Jules', 30.0), ('TD', 35.0)]


## 3. Function timing.

- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible


In [19]:
# Code here
import time
# Helper function to measure execution time of an RDD pipeline
def time_pipeline(pipeline_function, input_rdd):
    start_time = time.time()
    # take(1) forces Spark to execute the DAG pipeline
    pipeline_function(input_rdd).take(1)
    end_time = time.time()
    return end_time - start_time

In [20]:
# Pipeline 1: Unoptimized (we aggregate all words before any cleaning)
def unoptimized_pipeline(raw_rdd):
    return (
        raw_rdd
        .flatMap(lambda line: line.split(" "))
        .map(lambda word: (word.lower(), 1))
        .reduceByKey(lambda a, b: a + b)
    )

In [21]:
# Pipeline 2: Optimized (we clean and filter stopwords BEFORE reduceByKey)
def optimized_pipeline(raw_rdd):
    stopwords = {"the", "a", "an", "and", "or", "in", "on", "at", "to", "of"}
    return (
        raw_rdd
        .flatMap(lambda line: line.split(" "))
        .map(lambda word: word.lower())
        .filter(lambda word: len(word) > 0 and word not in stopwords)
        .map(lambda word: (word, 1))
        .reduceByKey(lambda a, b: a + b)
    )

In [22]:
# Measure and print the execution times
unoptimized_time = time_pipeline(unoptimized_pipeline, rdd)
optimized_time = time_pipeline(optimized_pipeline, rdd)

print(f"Unoptimized time: {unoptimized_time:.4f} seconds")
print(f"Optimized time:   {optimized_time:.4f} seconds")

Unoptimized time: 0.4231 seconds
Optimized time:   0.5942 seconds


## 4. Text Comparison

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two

In [23]:
# Code here

# 1. Download the original French text from Project Gutenberg
!wget -nc -O french_book.txt https://www.gutenberg.org/ebooks/46541.txt.utf-8

# 2. Load the French text file into an RDD
french_rdd = sc.textFile("/content/french_book.txt")

# 3. Define basic French stopwords
french_stopwords = {
    "le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "a","à",
    "dans", "il", "elle", "ils", "qui", "que", "ne", "pas", "se", "ce",
    "son", "sa", "ses", "au", "aux", "pour", "sur", "par", "avec", "d",
    "l", "qu", "plus", "tout", "mais", "comme", "cette", "si", "sans"
}

# 4. Clean and count the French text using our pipeline logic
french_counts = (
    french_rdd
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: word.lower().strip(string.punctuation))
    .filter(lambda word: len(word) > 0 and word not in french_stopwords)
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: x[1], ascending=False)
)

# 5. Display the top 10 most frequent words in English
print("Top 10 English words:")
for word, count in final_counts.take(10):
    print(f"{word}: {count}")

# 6. Display the top 10 most frequent words in French
print("\nTop 10 French words:")
for word, count in french_counts.take(10):
    print(f"{word}: {count}")

--2026-09-17 10:13:17--  https://www.gutenberg.org/ebooks/46541.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/46541/pg46541.txt [following]
URL transformed to HTTPS due to an HSTS policy
--2026-09-17 10:13:17--  https://www.gutenberg.org/cache/epub/46541/pg46541.txt
Reusing existing connection to www.gutenberg.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 472731 (462K) [text/plain]
Saving to: ‘french_book.txt’

french_book.txt     100%[===================>] 461.65K  --.-KB/s    in 0.1s    

2026-09-17 10:13:17 (3.74 MB/s) - ‘french_book.txt’ saved [472731/472731]

Top 10 English words:
fogg: 577
which: 515
passepartout: 392
mr: 373
him: 314
would: 278
phileas: 250
fix: 228
who: 199
said: 194

Top 10 French words:
fogg: 681
passep

Text Comparison and Exploratory Data Analysis:

Main Characters:
The core proper nouns match across both languages. Fogg, Passepartout, Phileas, and Fix are the most frequent words in both the English translation and the French original, appearing in almost the exact same rank order.

Language Nuances and Dialogue:
In French, pronouns such as lui and vous appear in the top ten, reflecting the formal dialogue between characters. In English, helper words like which, would, and said are more frequent due to the translator choices.

Thematic Consistency:
The presence of the word mr in both texts and the word heures in French highlights the British setting and the central plot of racing against time around the world.